# Drug Verification — Training Monitor

This notebook trains the PK/PD neural network controller and provides detailed monitoring of training behaviour for evaluation.

**Branch:** `feat/jess-suggested-vclScalar`  
**Repo:** [lstrsrmn/drug-verification](https://github.com/lstrsrmn/drug-verification)

### Sections
1. Setup
2. Data Generation
3. Preprocessing
4. Model Definition
5. Training with Monitoring
6. Evaluation
7. Diagnostic Plots

---

### Notes

D_prev was trialled as a 6th input feature. While it reduced Val MSE from ~54 to ~1.4, it caused safeFar and safeNear verification properties to fail — the verifier found physically impossible inputs (high D_prev, low C) that the network would overdose. D_prev has been removed and the model is back to 5 inputs: [C, T, WBC, Age, Weight].

A +0.0001 positive clamp is added to the ONNX graph at export so the network never outputs exactly zero, satisfying the nonNeg property.

> **Note:** If you have previously run this notebook in Colab, delete the `drug-verification` directory and re-run Setup to pick up the updated code.

## 1. Setup

In [ ]:
import os, sys

# Clone the repo if running in Colab
if 'google.colab' in sys.modules:
    if not os.path.exists('drug-verification'):
        !git clone --branch feat/jess-suggested-vclScalar https://github.com/lstrsrmn/drug-verification.git
    os.chdir('drug-verification')

    # Add src/ to path so drug_verification package is importable without editable install
    if os.path.abspath('src') not in sys.path:
        sys.path.insert(0, os.path.abspath('src'))

    # Install only the deps needed for training — skips vehicle-lang and torch
    !pip install -q tensorflow scikit-learn matplotlib pandas numpy

# Allow TF to grow GPU memory rather than reserving all of it
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # suppress info logs

print('Working directory:', os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from drug_verification.simulation import simulate_cohort
from drug_verification.training import build_model, prepare_data, evaluate_model
from drug_verification.types import SimulationConfig
from drug_verification import constants as C

print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## 2. Data Generation

Simulate a cohort of patients using the PK/PD model. All randomness is seeded for reproducibility.

In [ ]:
# --- Configuration --- edit these to experiment
NUM_PATIENTS = 50
TIMESTEPS    = 48
EPOCHS       = 50
BATCH_SIZE   = 32
SEED         = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
cfg = SimulationConfig(num_patients=NUM_PATIENTS, timesteps=TIMESTEPS, seed=SEED)
X, y = simulate_cohort(cfg, seed=SEED)

print(f'Samples:  {X.shape[0]}')
print(f'Features: {X.shape[1]}  (C, T, WBC, Age, Weight)')
print(f'Target:   dose (mg)')
print()
print('Feature ranges:')
feature_names = ['C (conc)', 'T (temp)', 'WBC', 'Age', 'Weight']
for i, name in enumerate(feature_names):
    print(f'  {name:12s}  min={X[:,i].min():.2f}  max={X[:,i].max():.2f}  mean={X[:,i].mean():.2f}')
print(f'  {"Dose":12s}  min={y.min():.2f}  max={y.max():.2f}  mean={y.mean():.2f}')

## 3. Preprocessing

Scaler is fit on the training split only to avoid data leakage.

In [ ]:
X_train, X_test, y_train, y_test, scaler = prepare_data(X, y, seed=SEED)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')
print()
print('Scaler mean: ', np.round(scaler.mean_, 4))
print('Scaler std:  ', np.round(scaler.scale_, 4))
print()
print('NOTE: These values must match meanScalingValues / standardDeviationValues in pk.vcl')
print('      They are auto-updated when you run `pk train` locally.')

## 4. Model Definition

In [ ]:
model = build_model(X_train.shape[1])
model.summary()

## 5. Training with Monitoring

Custom callback records per-epoch max absolute error alongside the standard loss metrics.

In [ ]:
class MaxAbsoluteErrorCallback(tf.keras.callbacks.Callback):
    """Records max absolute error on the validation set each epoch."""
    def __init__(self, X_val, y_val):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
        self.max_errors = []

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.X_val, verbose=0).flatten()
        max_err = float(np.max(np.abs(preds - self.y_val.flatten())))
        self.max_errors.append(max_err)
        print(f'  val_max_abs_err: {max_err:.2f} mg')

max_err_cb = MaxAbsoluteErrorCallback(X_test, y_test)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[max_err_cb],
)

## 6. Evaluation

In [ ]:
metrics = evaluate_model(model, X_test, y_test)
print('Test set evaluation')
print(f'  MSE:               {metrics["mse"]:.4f}')
print(f'  MAE:               {metrics["mae"]:.4f}')
print(f'  Max absolute error: {metrics["max_absolute_error"]:.4f} mg  ← worst-case overdose prediction')

## 7. Diagnostic Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
ax = axes[0]
ax.plot(history.history['loss'], label='Train loss')
ax.plot(history.history['val_loss'], label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('Loss curves')
ax.legend()
ax.grid(True, alpha=0.3)

# MAE curves
ax = axes[1]
ax.plot(history.history['mae'], label='Train MAE')
ax.plot(history.history['val_mae'], label='Val MAE')
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE (mg)')
ax.set_title('MAE curves')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(max_err_cb.max_errors, color='crimson', label='Val max absolute error')
plt.axhline(C.D_MAX * 0.1, color='orange', linestyle='--', label='10% of D_max (indicative threshold)')
plt.xlabel('Epoch')
plt.ylabel('Max absolute error (mg)')
plt.title('Worst-case prediction error per epoch')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
y_pred = model.predict(X_test, verbose=0).flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# True vs predicted
ax = axes[0]
ax.scatter(y_test.flatten(), y_pred, alpha=0.3, s=10)
lims = [0, max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1, label='Ideal')
ax.set_xlabel('True dose (mg)')
ax.set_ylabel('Predicted dose (mg)')
ax.set_title('True vs predicted dose')
ax.legend()
ax.grid(True, alpha=0.3)

# Prediction distribution
ax = axes[1]
ax.hist(y_pred, bins=40, alpha=0.6, label='Predicted', color='steelblue')
ax.hist(y_test.flatten(), bins=40, alpha=0.6, label='True', color='orange')
ax.set_xlabel('Dose (mg)')
ax.set_ylabel('Count')
ax.set_title('Prediction distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

zero_preds = np.sum(y_pred == 0)
print(f'Predictions at zero: {zero_preds} / {len(y_pred)} ({100*zero_preds/len(y_pred):.1f}%)  ← ReLU collapse check')

In [ ]:
residuals = y_pred - y_test.flatten()
feature_names = ['C (conc)', 'T (temp)', 'WBC', 'Age', 'Weight']

# Unscale X_test back to original space for interpretable axes
X_test_raw = scaler.inverse_transform(X_test)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (ax, name) in enumerate(zip(axes, feature_names)):
    ax.scatter(X_test_raw[:, i], residuals, alpha=0.3, s=8)
    ax.axhline(0, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel(name)
    ax.set_ylabel('Residual (mg)' if i == 0 else '')
    ax.set_title(f'Residuals vs {name}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Residuals by feature — systematic patterns indicate bias', y=1.02)
plt.tight_layout()
plt.show()

## 8. Verification

Export the trained model to ONNX, write the scaler `.idx` files, then run `vehicle verify` against all three safety properties:

| Property | What it checks |
|---|---|
| `nonNeg` | Output dose is always ≥ 0 |
| `safeFar` | When concentration is far from C_safe, the network does not push it over the limit |
| `safeNear` | When concentration is near C_safe, the network prescribes a near-zero dose |

**Prerequisites (local):** `vehicle` CLI and Marabou must be on your PATH.  
**Colab:** Run the install block below first.

In [ ]:
import subprocess, sys, os

# --- 1. Install vehicle CLI and download Marabou (Colab only) ---
if 'google.colab' in sys.modules:
    !pip install -q vehicle-lang

    # Download the pre-built Marabou binary for Linux x86-64
    # Check https://github.com/NeuralNetworkVerification/Marabou/releases for latest version
    MARABOU_VERSION = '2.0.0'
    MARABOU_URL = f'https://github.com/NeuralNetworkVerification/Marabou/releases/download/v{MARABOU_VERSION}/Marabou'
    if not os.path.exists('/usr/local/bin/Marabou'):
        !wget -q -O /usr/local/bin/Marabou {MARABOU_URL}
        !chmod +x /usr/local/bin/Marabou
    print('Marabou:', subprocess.run(['Marabou', '--version'], capture_output=True, text=True).stdout.strip() or 'installed')

# --- 2. Export model to ONNX ---
from drug_verification.training import export_onnx, update_vcl_scaler

ONNX_PATH = 'pk.onnx'
export_onnx(model, out_path=ONNX_PATH)

# --- 3. Write scaler .idx files (read by vehicle via -d flags) ---
update_vcl_scaler(scaler, spec_path='pk.vcl')
print('pk_mean.idx and pk_std.idx written')

# --- 4. Verify files are present ---
for f in [ONNX_PATH, 'pk.vcl', 'pk_mean.idx', 'pk_std.idx']:
    status = '✓' if os.path.exists(f) else '✗ MISSING'
    print(f'  {status}  {f}')

In [ ]:
import subprocess, shutil

PROPERTIES = ['nonNeg', 'safeNear', 'safeFar']
PARAMS = [
    '-p', 'Ka:4.5',
    '-p', 'Ke:3.5',
    '-p', 'Vd:10',
    '-p', 'C_safe:30',
    '-p', 'ttd:2',
    '-p', 'Ka_over:0.3228',
    '-p', 'Ka_under:0.3227',
    '-p', 'Ke_over:0.415',
    '-p', 'Ke_under:0.4149',
    '-p', 'eps:0.001',
]

if shutil.which('vehicle') is None:
    print('ERROR: vehicle not found on PATH — install it or add it to your PATH')
else:
    results = {}
    for prop in PROPERTIES:
        cmd = [
            'vehicle', 'verify',
            '-v', 'Marabou',
            '-s', 'pk.vcl',
            '-n', f'pk:{ONNX_PATH}',
            '-c', 'cache',
            '-d', 'meanScalingValues:pk_mean.idx',
            '-d', 'standardDeviationValues:pk_std.idx',
            '--property', prop,
        ] + PARAMS

        print(f'\nVerifying {prop} ...')
        result = subprocess.run(cmd, capture_output=True, text=True)
        output = (result.stdout + result.stderr).strip()
        print(output)

        passed = result.returncode == 0 and 'verified' in output.lower()
        results[prop] = 'PASS ✓' if passed else 'FAIL ✗'

    print('\n' + '='*40)
    print('Verification summary')
    print('='*40)
    for prop, status in results.items():
        print(f'  {prop:12s}  {status}')